In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
from scipy.stats import zscore
import time

# Load dataset
df= pd.read_csv("data/AmesHousing_engineered.csv")

# Drop target and ID columns
X = df.drop(columns=["SalePrice", "PID", "Order"], errors="ignore")

print("Scaled features shape:", X.shape)



Scaled features shape: (2930, 172)


In [4]:

#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [5]:
# Compute z-scores
z_scores = np.abs((X - X.mean()) / X.std())

# Define threshold
threshold = 3
# Get row indices where ANY feature is an outlier
outlier_indices = np.where((z_scores > threshold).any(axis=1))[0]

# Remove them
X_clean = X.drop(index=X.index[outlier_indices])
print("Original features shape:", X.shape)
print("After Outlier Removal:", X_clean.shape)


Original features shape: (2930, 172)
After Outlier Removal: (2766, 172)


In [6]:
#Apply StandardScaler
scaler = StandardScaler()
X_so = scaler.fit_transform(X_clean)
print("Scaled shape:", X_so.shape)

Scaled shape: (2766, 172)


In [7]:
#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [8]:
#K-Means on Scaled + OutlierRemoval Data
start_time = time.time()
kmean_outliers = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    kmean_outliers.append({"algorithm": "KMeans", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 5.235440015792847 seconds
K-Means runtime: 5.2354 seconds


In [9]:
#Gaussian Mixture (GMM)on Scaled + OutlierRemoval Data
start_time = time.time()
gmm_outliers = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    gmm_outliers.append({"algorithm": "GMM", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")  

Runtime: 157.12408638000488 seconds
GMM runtime: 157.1241 seconds


In [10]:
#Agglomerative Clustering on Scaled + OutlierRemoval Data
start_time = time.time()
agg_outliers = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    agg_outliers.append({"algorithm": "Agglomerative", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")  

Runtime: 4.0682127475738525 seconds
Agglomerative runtime: 4.0682 seconds


In [11]:
#Spectral Clustering on Scaled + OutlierRemoval Data
start_time = time.time()
spec_outliers = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    spec_outliers.append({"algorithm": "Spectral", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 6.315542697906494 seconds
Spectral runtime: 6.3155 seconds


In [12]:
#DBSCAN on Scaled + OutlierRemoval Data
start_time = time.time()
dbscan_outliers = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_so)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1:
        sil, db, ch = compute_metrics(X_so[mask], labels[mask])
        dbscan_outliers.append({"algorithm": "DBSCAN", "preprocessing": "OutlierRemoval", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 0.159193754196167 seconds
DBSCAN runtime: 0.1592 seconds


In [13]:
from sklearn.cluster import Birch
start_time = time.time()

birch_outliers = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_so)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X_so, labels)
        birch_outliers.append({
            "algorithm": "BIRCH",
            "preprocessing": "OutliersRemoval",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")

Runtime: 5.47456169128418 seconds
Birch runtime: 5.4746 seconds


In [14]:
from sklearn.cluster import OPTICS
start_time = time.time()

optics_outliers = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_so)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_so, labels)
        optics_outliers.append({
            "algorithm": "OPTICS",
            "preprocessing": "OutliersRemoval",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"OPTICS runtime: {runtime:.4f} seconds")


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


Runtime: 41.29910206794739 seconds
OPTICS runtime: 41.2991 seconds


In [ ]:
import csv


ames_results_outliers = (kmean_outliers + gmm_outliers + agg_outliers + spec_outliers + dbscan_outliers+birch_outliers + optics_outliers)
# Desired column order
keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/ames_data/ames_outliers.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ames_results_outliers)